In [ ]:
library(Seurat)
library(Matrix)
library(variancePartition)
library(BiocParallel)
library(limma)
library(ggplot2)
library(dplyr)
library(DESeq2)
library(edgeR)



In [ ]:
import numpy

In [ ]:
set.seed(42)

In [ ]:
# -------------------------------
# Config helper: point to ONE folder
# -------------------------------
make_cfg <- function(root_dir,
                     seurat_rds_name = "seurat_object.rds",
                     raw_counts_name = "raw_counts.csv",
                     logged_counts_name = "normalized_counts.csv",
                     features_name   = "features_counts.csv",
                     metadata_name   = "cell_metadata.csv",
                     coords_name     = "coords_xy.csv",
                     quint_labels_name = "Color_key.xlsm") {
  
  list(
    root_dir = root_dir,
    seurat_rds = file.path(root_dir, seurat_rds_name),
    raw_counts = file.path(root_dir, raw_counts_name),
    logged_counts = file.path(root_dir, logged_counts_name),
    features   = file.path(root_dir, features_name),
    metadata   = file.path(root_dir, metadata_name),
    coords     = file.path(root_dir, coords_name),
    quint_labels = file.path(root_dir, quint_labels_name)
  )
}

# -------------------------------
# Build Seurat with strict alignment
# -------------------------------
build_seurat_from_folder <- function(cfg, assay_name = "RNA") {
  
  message(paste0("Loading from: ", cfg$root_dir))
  
  # 1. Read Data
  # We assume row.names = 1 contains the Cell IDs for counts/meta/coords
  counts        <- read.csv(cfg$raw_counts, row.names = 1, check.names = FALSE)
  logged_counts <- read.csv(cfg$logged_counts, row.names = 1, check.names = FALSE)
  features      <- read.csv(cfg$features,   row.names = 1, check.names = FALSE)
  meta          <- read.csv(cfg$metadata,   row.names = 1, check.names = FALSE)
  coords        <- read.csv(cfg$coords,     row.names = 1, check.names = FALSE)
  
  # 2. Safety Check: Gene Dimensions
  # Verify the number of columns in counts matches the number of features
  if (ncol(counts) != nrow(features)) {
    stop(paste("Mismatch: Counts matrix has", ncol(counts), "genes, but features file has", nrow(features)))
  }
  
  # Apply gene names (features) to the columns
  colnames(counts) <- rownames(features)
  colnames(logged_counts) <- rownames(features)
  
  # 3. ALIGN SAMPLES (The Fix)
  # Find cells present in ALL required files (Counts, Meta, and Coords)
  common_cells <- intersect(rownames(counts), rownames(meta))
  common_cells <- intersect(common_cells, rownames(coords))
  common_cells <- intersect(common_cells, rownames(logged_counts))
  
  if (length(common_cells) == 0) {
    stop("Error: No common Cell IDs found between counts, metadata, and coords. Check your CSV row names.")
  }
  
  message(paste0("  Matched ", length(common_cells), " cells across all files."))
  
  # Subset and Reorder all files to match 'common_cells' exactly
  counts        <- counts[common_cells, ]
  logged_counts <- logged_counts[common_cells, ]
  meta          <- meta[common_cells, ]
  coords        <- coords[common_cells, ]
  
  # 4. Create Seurat Object
  # Seurat expects genes (rows) x cells (cols), so we transpose
  counts_seurat <- t(counts)
  logged_seurat <- t(logged_counts)
  
  seu <- CreateSeuratObject(
    counts    = counts_seurat,
    meta.data = meta,
    assay     = assay_name
  )
  
  # Add normalized data
  seu <- SetAssayData(seu, layer = "data", new.data = as.matrix(logged_seurat))
  
  # Add coords
  # Since we aligned 'coords' to 'common_cells' above, it matches perfectly
  seu <- AddMetaData(seu, metadata = coords)
  
  return(seu)
}

# -------------------------------
# Execution
# -------------------------------
# Make sure to set your WD if needed, though full paths in cfg avoid this issue
# setwd("/your/path/here")

print("1: Full")
cfg_file <- make_cfg("/path/to/csvs/base")
base <- build_seurat_from_folder(cfg_file)

#print("2: Cortex Up")
#cfg_file_up <- make_cfg("/path/to/csvs/up")
#up <- build_seurat_from_folder(cfg_file_up)

#print("3: Cortex Down")
#cfg_file_down <- make_cfg("/path/to/csvs/down")
#down <- build_seurat_from_folder(cfg_file_down)


In [ ]:
seurat_obj_no_O<- subset(base, subset = napari_region != "Olfactory")

In [ ]:
unique(seurat_obj_no_O$napari_region)

In [ ]:

# ==============================================================================
run_de_napari <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
    message(paste0("\n>>> [GLOBAL] STARTING DREAM FOR: ", dataset_name))
    
    # 1. Global Formula
    num_cts <- length(unique(seurat_obj$cell_type))
      
    if (num_cts > 1) {
        message(paste0("    Detected ", num_cts, " cell types. Including (1|cell_type)."))
        form_de <- ~ Treatment + log_depth + (1 | napari_region) + (1 | cell_type)  + (1|sample_ID)
    } else {
        message("    Detected Single Cell Type. Removing (1|ct_simple) from formula.")
        form_de <- ~ Treatment + log_depth + (1 | napari_region) + (1|sample_ID)
    }
  
    message(paste("    Formula:", deparse(form_de)))
    
    # 2. Prepare Data for Voom (Requires Raw Counts)
    counts <- as.matrix(GetAssayData(seurat_obj, layer = "counts"))
    info   <- seurat_obj@meta.data
    
    # Filter genes with 0 counts in this specific subset
    keep_genes <- rowSums(counts) > 0
    counts <- counts[keep_genes, ]
    
    # 3. Apply Voom with Dream Weights
    # This calculates precision weights for heteroscedasticity
    dge <- DGEList(counts)
    dge <- calcNormFactors(dge) # TMM Normalization
    
    message("    Calculating voom weights...")
    vobj <- voomWithDreamWeights(dge, form_de, info, BPPARAM = param)
    rm(counts, dge, seurat_obj); gc() 
    # 4. Fit Dream
    # Note: 'vobj' contains the weighted logCPM matrix
    fit <- dream(vobj, form_de, info, BPPARAM = param)
    fit <- eBayes(fit)
  
    # 5. Extract Treatment Results
    target_coef <- grep("Treatment", colnames(fit$coefficients), value = TRUE)
    target_coef <- target_coef[length(target_coef)] 
  
    if (length(target_coef) > 0) {
        de_res <- topTable(fit, coef = target_coef, number = Inf, sort.by = "P", confint = TRUE)
        de_res$Gene <- rownames(de_res)
        
        # --- Add Metrics ---
        # We use the raw counts to calculate % expressed, consistent with Seurat logic
        # (Re-matching rows because we filtered genes earlier)
        de_res$pct_1 <- rowMeans(counts[rownames(de_res), , drop=FALSE] > 0)
        
        outfile <- file.path(save_dir, paste0("Jan_26", out_prefix, ".csv"))
        write.csv(de_res, file = outfile, row.names = FALSE)
    }
}
# ==============================================================================
# FIXED: run_de_global_quint (Now uses Random Effects for Variance Partitioning)
# ==============================================================================
run_de_quint <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
    message(paste0("\n>>> [GLOBAL] STARTING DREAM FOR: ", dataset_name))
    
    # 1. Global Formula
    num_cts <- length(unique(seurat_obj$cell_type))
      
    if (num_cts > 1) {
        message(paste0("    Detected ", num_cts, " cell types. Including (1|cell_type)."))
        form_de <- ~ Treatment + log_depth + (1 | quint_region) + (1 | cell_type)  + (1|sample_ID)
    } else {
        message("    Detected Single Cell Type. Removing (1|ct_simple) from formula.")
        form_de <- ~ Treatment + log_depth + (1 | quint_region) + (1|sample_ID)
    }
  
    message(paste("    Formula:", deparse(form_de)))
    
    # 2. Prepare Data for Voom (Requires Raw Counts)
    counts <- as.matrix(GetAssayData(seurat_obj, layer = "counts"))
    info   <- seurat_obj@meta.data
    
    # Filter genes with 0 counts in this specific subset
    keep_genes <- rowSums(counts) > 0
    counts <- counts[keep_genes, ]
    
    # 3. Apply Voom with Dream Weights
    # This calculates precision weights for heteroscedasticity
    dge <- DGEList(counts)
    dge <- calcNormFactors(dge) # TMM Normalization
    
    message("    Calculating voom weights...")
    vobj <- voomWithDreamWeights(dge, form_de, info, BPPARAM = param)
    rm(counts, dge, seurat_obj); gc() 
    # 4. Fit Dream
    # Note: 'vobj' contains the weighted logCPM matrix
    fit <- dream(vobj, form_de, info, BPPARAM = param)
    fit <- eBayes(fit)
  
    # 5. Extract Treatment Results
    target_coef <- grep("Treatment", colnames(fit$coefficients), value = TRUE)
    target_coef <- target_coef[length(target_coef)] 
  
    if (length(target_coef) > 0) {
        de_res <- topTable(fit, coef = target_coef, number = Inf, sort.by = "P", confint = TRUE)
        de_res$Gene <- rownames(de_res)
        
        # --- Add Metrics ---
        # We use the raw counts to calculate % expressed, consistent with Seurat logic
        # (Re-matching rows because we filtered genes earlier)
        de_res$pct_1 <- rowMeans(counts[rownames(de_res), , drop=FALSE] > 0)
        
        outfile <- file.path(save_dir, paste0("Jan_26", out_prefix, ".csv"))
        write.csv(de_res, file = outfile, row.names = FALSE)
    }
}
###############################################################################################################################################
###############################################################################################################################################
###############################################################################################################################################
###############################################################################################################################################

run_de_blind <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
  
    num_cts <- length(unique(seurat_obj$cell_type))                    
      
    if (num_cts > 10) {
        # CASE A: Whole Brain (Many Cell Types)
        message(paste0("    Detected ", num_cts, " cell types. Including (1|ct_code) and (1|quint_region)."))
        # CHANGED: quint_region is now (1|quint_region)
        form_de <- ~ Treatment + log_depth  + (1 | cell_type)   + (1|sample_ID)    
    } else {
        # CASE B: Single Cell Type
        message("    Detected Single Cell Type. Removing (1|ct_code). Keeping (1|quint_region).")
        # CHANGED: quint_region is now (1|quint_region)
        form_de <- ~ Treatment + log_depth  + (1|sample_ID)
    }
    form_de <- ~ Treatment + log_depth  + (1|sample_ID)
    message(paste0("    >>> [LOCAL] Fitting Dream for: ", dataset_name))
  
    # Pre-fetch data for metrics calculation later
    # We keep 'geneExpr' (normalized) for the final average stats if desired, 
    # but use 'counts' for the model.
    geneExpr <- as.matrix(GetAssayData(seurat_obj, layer = "data"))
    counts   <- as.matrix(GetAssayData(seurat_obj, layer = "counts"))
    info     <- seurat_obj@meta.data
  
    tryCatch({
        # 2. Prepare Data for Voom
        keep_genes <- rowSums(counts) > 0
        counts_sub <- counts[keep_genes, ]
        
        dge <- DGEList(counts_sub)
        dge <- calcNormFactors(dge)
        
        # 3. Apply Voom

        vobj <- voomWithDreamWeights(dge, form_de, info, BPPARAM = param)
        rm(counts, dge, seurat_obj); gc() 
        # 4. Fit Dream
        fit <- dream(vobj, form_de, info, BPPARAM = param)
        fit <- eBayes(fit)
        
        target_coef <- grep("Treatment", colnames(fit$coefficients), value = TRUE)
        target_coef <- target_coef[length(target_coef)]
        
        if (length(target_coef) > 0) {
            de_res <- topTable(fit, coef = target_coef, number = Inf, sort.by = "P", confint = TRUE)
            de_res$Gene <- rownames(de_res)
            
            # --- Add Critical ScRNA-seq Metrics ---
            ref_level <- levels(seurat_obj$Treatment)[1]
            target_level <- levels(seurat_obj$Treatment)[2]
            cells_ref <- which(seurat_obj$Treatment == ref_level)
            cells_target <- which(seurat_obj$Treatment == target_level)
            
            # Use original normalized matrix for these stats (matches previous logic)
            de_res$pct_ref    <- rowMeans(geneExpr[rownames(de_res), cells_ref, drop=FALSE] > 0)
            de_res$pct_target <- rowMeans(geneExpr[rownames(de_res), cells_target, drop=FALSE] > 0)
            de_res$avg_ref    <- rowMeans(geneExpr[rownames(de_res), cells_ref, drop=FALSE])
            de_res$avg_target <- rowMeans(geneExpr[rownames(de_res), cells_target, drop=FALSE])
            
            outfile <- file.path(save_dir, paste0("Jan_26", out_prefix, ".csv"))
            write.csv(de_res, file = outfile, row.names = FALSE)
            message(paste("    -> Saved:", outfile))
        }
    }, error = function(e) {
        message(paste("    !! SKIPPING: Model failed (likely low variance/cells):", e$message))
    })
}

###############################################################################################################################################
###############################################################################################################################################
###############################################################################################################################################
###############################################################################################################################################

run_pseudobulk_deseq2 <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
   
  message(paste0("    >>> [PB-VAL] Running Pseudo-bulk DESeq2: ", dataset_name))
  
  # 1. Aggregate Counts by Sample
  cts <- AggregateExpression(seurat_obj, group.by = "sample_ID", assays = "RNA", slot = "counts")$RNA
  colnames(cts) <- gsub("^g", "", colnames(cts))
  colnames(cts) <- gsub("-", "_", colnames(cts))
  
  # 2. Create Sample Metadata
  colData <- seurat_obj@meta.data %>%
    select(sample_ID, Treatment) %>%
    distinct(sample_ID, .keep_all = TRUE)

  # Ensure order matches columns of cts
  colData <- colData[match(colnames(cts), colData$sample_ID), ]
  rownames(colData) <- colData$sample_ID
  
  # 3. Run DESeq2
  tryCatch({
    dds <- DESeqDataSetFromMatrix(countData = cts, colData = colData, design = ~ Treatment)
    
    # Pre-filtering (keep genes with > 10 counts total)
    keep <- rowSums(counts(dds)) >= 10
    dds <- dds[keep,]
    
    dds <- DESeq(dds, quiet = TRUE)
    
    # 4. Get Results
    res <- results(dds)
    res_df <- as.data.frame(res)
    res_df$Gene <- rownames(res_df)
    
    outfile <- file.path(save_dir, paste0("Jan_26_DESEQ2", out_prefix, ".csv"))
    write.csv(res_df, file = outfile, row.names = FALSE)
    message(paste("    -> Saved PB Validation:", outfile))
    
  }, error = function(e) {
    message(paste("    !! PB FAILED: Likely not enough samples/replicates.", e$message))
  })
}

# ==============================================================================
# HELPER: PREPARE METADATA (Run this on the object first)
# ==============================================================================
prepare_metadata <- function(seurat_obj) {
  # 1. Scale sequencing depth (Vital for LMM convergence)
  names(seurat_obj@meta.data)[names(seurat_obj@meta.data) == "FMT"] <- "Treatment"
  seurat_obj$log_depth <- as.numeric(scale(log10(seurat_obj$nFeature_RNA + 1)))
  seurat_obj <- subset(seurat_obj, subset = Treatment != "Cntrl")
  
  
  # 2. Ensure factors
  seurat_obj$Treatment <- as.factor(seurat_obj$Treatment)
  
  # 3. Relevel if possible
  if ("Healthy_FMT" %in% levels(seurat_obj$Treatment)) {
    seurat_obj$Treatment <- relevel(seurat_obj$Treatment, ref = "Healthy_FMT")
  }
  return(seurat_obj)
}# Remove 'Cntrl' from the object
# ==============================================================================
# ==============================================================================
# ==============================================================================                    ######################
# SEURAT ALTERNATIVE: run_seurat_global (Replaces run_de_napari / quint)        ##########################################
# ==============================================================================                    ######################
# ==============================================================================
# ==============================================================================


run_seurat_napari <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
    message(paste0("\n>>> [SEURAT + LATENT] STARTING FINDMARKERS FOR: ", dataset_name))
    
    # 1. Setup Idents
    Idents(seurat_obj) <- "Treatment"
    ref_level <- levels(seurat_obj$Treatment)[1]
    target_level <- levels(seurat_obj$Treatment)[2]
    
    # 2. Define Covariates (Latent Variables)
    # We mirror your LMM formula: ~ Treatment + log_depth + napari_region
    covariates <- c("log_depth")
    
    if ("napari_region" %in% colnames(seurat_obj@meta.data)) {
        message("    Including 'napari_region' as a latent variable.")
        # Ensure it is treated as a factor if it isn't already
        seurat_obj$napari_region <- as.factor(seurat_obj$napari_region)
        covariates <- c("log_depth", "napari_region")
    }
    else{
        message("Couldn't find it")
        }
    
    message("covatirates are:",covariates)
    tryCatch({
        de_res <- FindMarkers(seurat_obj, 
                              ident.1 = target_level, 
                              ident.2 = ref_level,
                              test.use = "LR",   
                              latent.vars = covariates, # <--- THIS IS THE KEY CHANGE
                              logfc.threshold = 0,   
                              min.pct = 0,          
                              verbose = FALSE)
        
        # 4. Formatting
        de_res$Gene <- rownames(de_res)
        colnames(de_res)[colnames(de_res) == "pct.1"] <- "pct_target"
        colnames(de_res)[colnames(de_res) == "pct.2"] <- "pct_ref"
        colnames(de_res)[colnames(de_res) == "avg_log2FC"] <- "logFC" 
        colnames(de_res)[colnames(de_res) == "p_val_adj"] <- "adj.P.Val"
        colnames(de_res)[colnames(de_res) == "p_val"] <- "P.Value"

        # 5. Save Results
        outfile <- file.path(save_dir, paste0("Jan_26", out_prefix, ".csv"))
        write.csv(de_res, file = outfile, row.names = FALSE)
        message(paste("    -> Saved Seurat (Latent Napari):", outfile))
        
    }, error = function(e) {
        message(paste("    !! SEURAT FAILED:", e$message))
    })
}

# ==============================================================================
# SEURAT GLOBAL: With Latent Variables (Quint Region)
# ==============================================================================
run_seurat_quint <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
    message(paste0("\n>>> [SEURAT + LATENT] STARTING FINDMARKERS FOR: ", dataset_name))
    
    Idents(seurat_obj) <- "Treatment"
    ref_level <- levels(seurat_obj$Treatment)[1]
    target_level <- levels(seurat_obj$Treatment)[2]
    
    # Define Covariates
    covariates <- c("log_depth")
    
    if ("quint_region" %in% colnames(seurat_obj@meta.data)) {

        # Ensure it is treated as a factor if it isn't already
        seurat_obj$quint_region <- as.factor(seurat_obj$quint_region)
        covariates <- c("log_depth", "quint_region")
    }
    
    else{
        message("Couldn't find it")
        }
    message("covatirates are:",covariates)
    tryCatch({
        de_res <- FindMarkers(seurat_obj, 
                              ident.1 = target_level, 
                              ident.2 = ref_level,
                              test.use = "LR",   
                              latent.vars = covariates, # <--- Regressing out Quint Region
                              logfc.threshold = 0,   
                              min.pct = 0,          
                              verbose = FALSE)
        
        de_res$Gene <- rownames(de_res)
        colnames(de_res)[colnames(de_res) == "pct.1"] <- "pct_target"
        colnames(de_res)[colnames(de_res) == "pct.2"] <- "pct_ref"
        colnames(de_res)[colnames(de_res) == "avg_log2FC"] <- "logFC" 
        colnames(de_res)[colnames(de_res) == "p_val_adj"] <- "adj.P.Val"
        colnames(de_res)[colnames(de_res) == "p_val"] <- "P.Value"

        outfile <- file.path(save_dir, paste0("Jan_26", out_prefix, ".csv"))
        write.csv(de_res, file = outfile, row.names = FALSE)
        message(paste("    -> Saved Seurat (Latent Quint):", outfile))
        
    }, error = function(e) {
        message(paste("    !! SEURAT FAILED:", e$message))
    })
}

run_seurat_blind <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
    message(paste0("\n>>> [SEURAT + LATENT] STARTING FINDMARKERS FOR: ", dataset_name))
    
    Idents(seurat_obj) <- "Treatment"
    ref_level <- levels(seurat_obj$Treatment)[1]
    target_level <- levels(seurat_obj$Treatment)[2]
    
    # Define Covariates
    covariates <- c("log_depth")
    message("covatirates are:",covariates)
    tryCatch({
        de_res <- FindMarkers(seurat_obj, 
                              ident.1 = target_level, 
                              ident.2 = ref_level,
                              test.use = "LR",   
                              latent.vars = covariates, # <--- Regressing out Quint Region
                              logfc.threshold = 0,   
                              min.pct = 0,          
                              verbose = FALSE)
        
        de_res$Gene <- rownames(de_res)
        colnames(de_res)[colnames(de_res) == "pct.1"] <- "pct_target"
        colnames(de_res)[colnames(de_res) == "pct.2"] <- "pct_ref"
        colnames(de_res)[colnames(de_res) == "avg_log2FC"] <- "logFC" 
        colnames(de_res)[colnames(de_res) == "p_val_adj"] <- "adj.P.Val"
        colnames(de_res)[colnames(de_res) == "p_val"] <- "P.Value"

        outfile <- file.path(save_dir, paste0("Jan_26", out_prefix, ".csv"))
        write.csv(de_res, file = outfile, row.names = FALSE)
        message(paste("    -> Saved Seurat (blind):", outfile))
        
    }, error = function(e) {
        message(paste("    !! SEURAT FAILED:", e$message))
    })
}
run_wilcox <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
    message(paste0("\n>>> [SEURAT + LATENT] STARTING FINDMARKERS FOR: ", dataset_name))
    
    Idents(seurat_obj) <- "Treatment"
    ref_level <- levels(seurat_obj$Treatment)[1]
    target_level <- levels(seurat_obj$Treatment)[2]
    
    # Define Covariates
    covariates <- c("log_depth")

    tryCatch({
        de_res <- FindMarkers(seurat_obj, 
                              ident.1 = target_level, 
                              ident.2 = ref_level,
                              test.use = "wilcox",   
                              #latent.vars = covariates, # <--- Regressing out Quint Region
                              logfc.threshold = 0,   
                              min.pct = 0,          
                              verbose = FALSE)
        
        de_res$Gene <- rownames(de_res)
        colnames(de_res)[colnames(de_res) == "pct.1"] <- "pct_target"
        colnames(de_res)[colnames(de_res) == "pct.2"] <- "pct_ref"
        colnames(de_res)[colnames(de_res) == "avg_log2FC"] <- "logFC" 
        colnames(de_res)[colnames(de_res) == "p_val_adj"] <- "adj.P.Val"
        colnames(de_res)[colnames(de_res) == "p_val"] <- "P.Value"

        outfile <- file.path(save_dir, paste0("Jan_26", out_prefix, ".csv"))
        write.csv(de_res, file = outfile, row.names = FALSE)
        message(paste("    -> Saved Wilcoxon:", outfile))
        
    }, error = function(e) {
        message(paste("    !! SEURAT FAILED:", e$message))
    })
}

In [ ]:
# ==============================================================================
# 0. SETUP: LIBRARIES & DIRECTORIES
# ==============================================================================
library(DESeq2)

  param <- SnowParam(workers = 5, type = "SOCK",
                     progressbar = TRUE, exportglobals = FALSE)


base_dir   <- "/path/to/lmm_outputs/base"
dirs       <- list(
  global = file.path(base_dir, "Global_CT_Analysis"),
  local  = file.path(base_dir, "Local_Regional_Analysis"),
  pb     = file.path(base_dir, "Pseudobulk_Validation")
)
sapply(dirs, function(x) if(!dir.exists(x)) dir.create(x, recursive=TRUE))

# ==============================================================================
# 1. HELPER: WRAPPER FOR ANALYSIS SUITE
#    Runs all 4 models (PB, Local, Global, Quint) on a given object
# ==============================================================================
run_analysis_suite <- function(obj, label, file_tag) {
  
 run_de_blind(obj, 
             dataset_name = paste0(label, "_dream_blind"), 
              out_prefix   = paste0(label, "_", file_tag, "_dream_blind"), 
              save_dir     = dirs$local)
  
 # C. Global (Region Aware)
 run_de_napari(obj, 
               dataset_name = paste0(label, "_dream_napari"), 
               out_prefix   = paste0(label, "_", file_tag, "_dream_napari"), 
               save_dir     = dirs$global)
  
 # D. Global Quint (Spatial Quintiles)
 run_de_quint(obj, 
                    dataset_name = paste0(label, "_dream_quint"), 
                    out_prefix   = paste0(label, "_", file_tag, "_dream_quint"), 
                    save_dir     = dirs$global)
 #run_seurat_napari(obj, 
 #               dataset_name = paste0(label, "_seurat_napari"), 
 #               out_prefix   = paste0(label, "_", file_tag, "_seurat_napari"), 
 #               save_dir     = dirs$global)
 
  # D. Global Quint (Spatial Quintiles)
 #run_seurat_quint(obj, 
 #                     dataset_name = paste0(label, "_seurat_quint"), 
 #                     out_prefix   = paste0(label, "_", file_tag, "_seurat_quint"), 
 #                     save_dir     = dirs$global)

 #run_seurat_blind(obj, 
 #                     dataset_name = paste0(label, "_seurat_blind"), 
 #                     out_prefix   = paste0(label, "_", file_tag, "_seaurat_blind"), 
 #                     save_dir     = dirs$global)
}

target_cell_types <- c("Astrocytes.cortex.hippocampus", "Astrocytes", "Microglia")

run_ct_loop <- function(data_obj, dataset_label) {
  
  message(paste0("\n>>> STARTING HIERARCHICAL LOOP: ", dataset_label))
  
  for (ct in target_cell_types) {
    # Dynamic column selection logic
    cell_col <- if (ct == "Astrocytes.cortex.hippocampus") "cell_type" else "ct_simple"
    #subset
    obj_ct <- data_obj[, data_obj@meta.data[[cell_col]] == ct]
    
    if (ncol(obj_ct) < 100) { message(paste("Skipping low cell count:", ct)); next }
    
    ct_clean <- gsub("[^A-Za-z0-9]", "_", ct)
    message(paste0("   Processing: ", ct))
    
    run_analysis_suite(obj_ct, 
                       label    = paste0(dataset_label, "_", ct), 
                       file_tag = ct_clean)
    
    rm(obj_ct); gc()
  }
}

# Execu# ==============================================================================
seurat_obj_no_O   <- prepare_metadata(seurat_obj_no_O)


message("\n>>> RUNNING WHOLE DATASETS...")
run_analysis_suite(seurat_obj_no_O,   label = "BASE",   file_tag = "WHOLE")

message("\n>>> RUNNING CT_loop DATASETS...")
run_ct_loop(seurat_obj_no_O, "BASE")


message("\n--- ALL ANALYSES COMPLETE ---")